## Topic: Parallel Chain in LangChain

### Agenda
- 1. Introduction of Parallel Chain

- 2. Architecture of Parallel Chain

- 3. What is RunnableParallel?

- 4. Real-World Example

- 5. 5. Complete Summary

### 1. Introduction of Parallel Chain
- Definition:
    - Parallel Chains execute multiple independent sub-chains simultaneously (at the same time) instead of one after another. All sub-chains receive the same input, and their outputs are merged into a single dictionary.


- Key idea:
    - Sequential Chain is based on dependency, while Parallel Chain is based on independence


- Key Points for Why Do We Need Parallel Chains?
    - 1. Wasted Time in Sequential Execution

    - 2. Give the better User Experience.

    - 3. Cost Efficiency


- Key NOTE:
    - RunnableParallel allows multiple independent LangChain Runnables to process the input concurrently and returns their results together as a dictionary

In [ ]:
# The core idea
""" 
SEQUENTIAL (One at a time):
──────────────────────────
Input → [Chain A: 3s] → [Chain B: 3s] → [Chain C: 3s] → Output
Total Time: 9 seconds 


PARALLEL (All at once):
───────────────────────
         ┌→ [Chain A: 3s] ─┐
Input ───┼→ [Chain B: 3s] ─┼→ Merged Output
         └→ [Chain C: 3s] ─┘
Total Time: 3 seconds  (3x faster!)

"""

In [ ]:
"""  - When Parallel Chain use: 

┌─────────────────────────────────────────────────────────────┐
│            USE PARALLEL CHAINS WHEN:                        │
├─────────────────────────────────────────────────────────────┤
│ -  Sub-chains are INDEPENDENT (don't need each other's      │
│    output)                                                  │
├─────────────────────────────────────────────────────────────┤
│ -  Same input needs to be analyzed from multiple angles     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ -  You want to compare outputs from different models        │
├─────────────────────────────────────────────────────────────┤
│ -  You need to extract multiple types of data from one text │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ -  Latency matters (web apps, chatbots, APIs)               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ -  You're doing fan-out / fan-in patterns                   │
└─────────────────────────────────────────────────────────────┘

"""

"""   - When NOT to Use:
┌─────────────────────────────────────────────────────────────┐
│          DON'T USE PARALLEL WHEN:                           │
├─────────────────────────────────────────────────────────────┤
│   Step B depends on Step A's output (use sequential)        │
├─────────────────────────────────────────────────────────────┤
│   Steps modify shared state (race conditions)               │
├─────────────────────────────────────────────────────────────┤
│   You're rate-limited by the API (parallel = burst of calls)│
│   Order of execution matters                                │
└─────────────────────────────────────────────────────────────┘ 


"""

### 2. Architecture of Parallel Chain

In [ ]:
""" 
                  {"topic": "Generative AI"}
                             │
              ┌──────────────┼──────────────┐
              ↓              ↓              ↓
        Explanation      Advantages    Disadvantages
          Chain             Chain          Chain
              ↓              ↓              ↓
        Explanation      Advantages    Disadvantages
              └──────────────┼──────────────┘
                             ↓
                    RunnableParallel
                             ↓
                     Combined Dictionary
"""

In [ ]:
"""  - Sequential vs Parallel — Visual Comparison

┌─────────────────────────────────────────────────────────────────┐
│                    SEQUENTIAL CHAIN                             │
│                                                                 │
│  Input ──► [Summarize] ──► [Sentiment] ──► [Keywords] ──► Out   │
│            (2s)           (2s)            (2s)                  │
│                                                                 │
│  Total: 6 seconds                                               │
│  Output: "keywords result" (only last step's output)            │
│  Use when: Each step DEPENDS on the previous                    │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                     PARALLEL CHAIN                              │
│                                                                 │
│              ┌──► [Summarize] (2s) ──► summary                  │
│              │                                                  │
│  Input ──────┼──► [Sentiment] (2s) ──► sentiment    ──► Merged  │
│              │                                     │    Output  │
│              └──► [Keywords]  (2s) ──► keywords    │            │
│                                                     │            │
│  Total: 2 seconds                                   │            │
│  Output: {summary: "...", sentiment: "...",         │            │
│           keywords: "..."}  ← ALL results!          │            │
└─────────────────────────────────────────────────────────────────┘


"""

### 3. What is RunnableParallel?

- Definition of RunnableParallel: 
    - RunnableParallel is a LangChain Runnable that takes a dictionary of named sub-chains, runs them all simultaneously on the same input, and returns a dictionary of named outputs.

    - execute the multiple chain at a time.

- for execute the parallel chain then import RunnableParallel from LangChain
    - from langchain_core.runnables import RunnableParallel

In [ ]:
### How RunnableParallel Works Internally
""" 
┌─────────────────────────────────────────────────────────────┐
│          RunnableParallel INTERNAL FLOW                     │
│                                                             │
│  1. RECEIVE INPUT                                           │
│     input = "LangChain is a framework for LLM apps."        │
│                                                             │
│  2. FAN-OUT: Send input to ALL sub-chains simultaneously    │
│     ┌──► summary_chain.invoke(input)   → Thread 1           │
│     ├──► sentiment_chain.invoke(input) → Thread 2           │
│     └──► keywords_chain.invoke(input)  → Thread 3           │
│                                                             │
│  3. WAIT: Wait for the SLOWEST chain to finish              │
│     Thread 1 finishes in 2.1s                               │
│     Thread 2 finishes in 1.8s                               │
│     Thread 3 finishes in 2.5s  ← Slowest!                   │
│     Total wait: 2.5s (not 6.4s!)                            │
│                                                             │
│  4. FAN-IN: Collect all results into a dictionary           │
│     output = {                                              │
│         "summary": "LangChain helps build LLM apps.",       │
│         "sentiment": "Positive",                            │
│         "keywords": ["LangChain", "LLM", "framework"]       │
│     }                                                       │
│                                                             │
│  5. RETURN the merged dictionary                            │
└─────────────────────────────────────────────────────────────┘

"""

### 4. Real-World Example

#### Example 1:
- idea:
    - prompt1 : text of details {domain}
        - domain: Machine learning

    - LLM1 : 
        - input: prompt1
        - process: Generate Note  base on {domain}
        - response1: Note

    - LLM2: 
        - input: prompt1
        - Process: Generate quiz base on {domain}
        - response2: quiz

    - LLM3:
        - input: LLM1, LLM2 -> response
        - process: Combine or merge response1 and response2
        - output: Note, quiz

In [ ]:
# Example 1:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

# Define model 1
model1 = ChatOpenAI()

# Define model 2
model2 = ChatAnthropic(
    model_name='claude-3-7-sonnet-20250219'
)


# Prompt1 -> notes
prompt1 = PromptTemplate(
    input_variables=["domain"],
    template = "Generate short and simple notes from the following domain {domain}"
)

# Prompt2 -> quiz
prompt2 = PromptTemplate(
    input_variables=["quiz"],
    template = "Generate 5 short question answer from the following domain \n {quiz}"
)


# Prompt3 -> Merge (notes, quiz)
prompt3 = PromptTemplate(
    input_variables=["notes", "quiz"],
    template = "Merge the Provided notes and quiz into single document \n notes ->{notes} and quiz -> {quiz}"
)

# Define the parser
parser = StrOutputParser()

# Collect all results into a dictionary -> using RunnableParallel
parallel_chain = RunnableParallel({
    # define the notes chain
    "notes" : prompt1 | model1 | parser,

    # define the quiz chain
    "quiz" : prompt2 | model2 | parser
})

# define a merge chain
merge_chain = prompt3 | model1 | parser


# Merger the chain (parallel_chain and merge_chain)
chain = parallel_chain | merge_chain


# another approach Define a domain text 
# domain = """ 

# """

# response
response = chain.invoke({
    "domain": "RAG"
})


print(response)

# for visualization the chain
chain.get_graph().print_ascii()

In [ ]:
# Real-World Example 1 — Multi-Aspect Document Analyzer
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
parser = StrOutputParser()

# ─── Chain 1: Executive Summary ───
summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are an executive briefing specialist."),
        ("human", "Summarize this document in exactly 3 sentences:\n\n{text}")
    ])
    | llm | parser
)

# ─── Chain 2: Key Entities Extraction ───
class Entities(BaseModel):
    people: list[str] = Field(description="People mentioned")
    organizations: list[str] = Field(description="Companies/organizations")
    locations: list[str] = Field(description="Places mentioned")
    dates: list[str] = Field(description="Important dates")

entity_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Extract named entities.\n{format_instructions}"),
        ("human", "{text}")
    ])
    | llm | JsonOutputParser()
)

# ─── Chain 3: Sentiment & Tone Analysis ───
tone_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Analyze the tone and sentiment."),
        ("human", "Analyze this text. Return: Overall Sentiment (Positive/Negative/Neutral), "
                  "Tone (Formal/Casual/Technical/Urgent), Confidence (0-100%):\n\n{text}")
    ])
    | llm | parser
)

# ─── Chain 4: Risk Detection ───
risk_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a risk analyst."),
        ("human", "Identify any risks, red flags, or concerns in this text. "
                  "If none, say 'No risks detected.':\n\n{text}")
    ])
    | llm | parser
)

# ─── Chain 5: Action Items ───
action_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a project manager."),
        ("human", "Extract actionable items from this text. "
                  "List them as numbered tasks. If none, say 'No action items.':\n\n{text}")
    ])
    | llm | parser
)

# ─── Combine ALL 5 in Parallel ───
document_analyzer = RunnableParallel(
    executive_summary=summary_chain,
    entities=entity_chain,
    sentiment_and_tone=tone_chain,
    risk_analysis=risk_chain,
    action_items=action_chain
)

# ─── Analyze a Document ───
document = """
Q3 2025 EARNINGS REPORT — TechGlobal Inc.

TechGlobal Inc. reported Q3 revenue of $2.4 billion, a 15%% increase 
year-over-year, driven by strong cloud services growth. CEO Sarah Chen 
highlighted the company's AI division as the fastest-growing segment, 
contributing $340M in new revenue.

However, the company faces a pending lawsuit from competitor DataFlow 
Corp alleging patent infringement in their machine learning pipeline. 
The trial is scheduled for January 2026 in the Northern District of 
California. Legal experts estimate potential damages of $200-500M.

CFO Michael Torres announced a $500M share buyback program and confirmed 
plans to open a new data center in Singapore by Q2 2026. The board has 
also approved hiring 500 additional AI engineers across the San Francisco, 
London, and Bangalore offices.

Key risks include supply chain disruptions affecting GPU procurement and 
increasing regulatory scrutiny of AI practices in the EU under the new 
AI Act, effective March 2026.
"""

# Run all 5 analyses SIMULTANEOUSLY
result = document_analyzer.invoke({"text": document})

# Display results
print("=" * 60)
print("  DOCUMENT ANALYSIS REPORT")
print("=" * 60)

print("\n  EXECUTIVE SUMMARY:")
print(result["executive_summary"])

print("\n  ENTITIES:")
entities = result["entities"]
print(f"   People: {entities.get('people', [])}")
print(f"   Organizations: {entities.get('organizations', [])}")
print(f"   Locations: {entities.get('locations', [])}")
print(f"   Dates: {entities.get('dates', [])}")

print("\n  SENTIMENT & TONE:")
print(result["sentiment_and_tone"])

print("\n  RISK ANALYSIS:")
print(result["risk_analysis"])

print("\n  ACTION ITEMS:")
print(result["action_items"])

In [ ]:
# Real-World Example 2 — E-Commerce Product Enrichment
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ─── Chain 1: SEO Optimization ───
seo_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are an e-commerce SEO expert."),
        ("human", "Generate SEO-optimized content for this product:\n{product}\n\n"
                  "Return JSON: {{\"seo_title\": \"...\", \"meta_description\": \"...\", "
                  "\"keywords\": [\"...\", \"...\"]}}")
    ])
    | llm | JsonOutputParser()
)

# ─── Chain 2: Marketing Copy ───
copy_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a copywriter for a luxury e-commerce brand."),
        ("human", "Write compelling marketing copy for:\n{product}\n\n"
                  "Include: Tagline (max 10 words), Description (50 words), "
                  "3 bullet-point selling points.")
    ])
    | llm | StrOutputParser()
)

# ─── Chain 3: Competitor Analysis ───
competitor_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a market research analyst."),
        ("human", "For this product, identify likely competitors and "
                  "suggested pricing strategy:\n{product}\n\n"
                  "Return: 3 competitor products, price range, positioning advice.")
    ])
    | llm | StrOutputParser()
)

# ─── Chain 4: Customer FAQ ───
faq_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a customer support specialist."),
        ("human", "Generate 5 frequently asked questions and answers "
                  "for this product:\n{product}")
    ])
    | llm | StrOutputParser()
)

# ─── Combine All 4 in Parallel ───
product_enricher = RunnableParallel(
    seo=seo_chain,
    marketing_copy=copy_chain,
    competitor_analysis=competitor_chain,
    faq=faq_chain
)

# ─── Enrich a Product ───
result = product_enricher.invoke({
    "product": "Sony WH-1000XM5 Wireless Noise-Cancelling Headphones, "
               "Black, 30-hour battery, Bluetooth 5.2, $348"
})

print(" SEO:")
print(f"   Title: {result['seo'].get('seo_title', 'N/A')}")
print(f"   Meta: {result['seo'].get('meta_description', 'N/A')}")
print(f"   Keywords: {result['seo'].get('keywords', [])}")

print("\n MARKETING COPY:")
print(result["marketing_copy"])

print("\n COMPETITOR ANALYSIS:")
print(result["competitor_analysis"])

print("\n FAQ:")
print(result["faq"]) 

### 5. Complete Summary


In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                    PARALLEL CHAINS                                │
│                                                                  │
│  WHAT:  Run multiple independent chains simultaneously           │
│  WHY:   Speed (3-7x faster), better UX, efficient processing    │
│  HOW:   RunnableParallel or dict shorthand                       │
│                                                                  │
│  SYNTAX:                                                         │
│    parallel = {                                                  │
│        "summary": summary_chain,                                 │
│        "sentiment": sentiment_chain,                             │
│        "keywords": keywords_chain                                │
│    }                                                             │
│    result = parallel.invoke({"text": "..."})                     │
│    # result = {"summary": "...", "sentiment": "...", ...}        │
│                                                                  │
│  PERFORMANCE:                                                    │
│    Sequential: Sum of all step times (N × T)                     │
│    Parallel:   Max of all step times (T)                         │
│    Speedup:    Typically 2x-7x faster                            │
│                                                                  │
│  KEY PATTERNS:                                                   │
│    ├── Fan-out / Fan-in (analyze from multiple angles)           │
│    ├── Multi-model consensus (compare LLM outputs)               │
│    ├── Parallel retrieval (search multiple sources)              │
│    ├── Sequential → Parallel → Sequential (hybrid)               │
│    └── Dynamic parallel (variable number of chains)              │
│                                                                  │
│  CAUTIONS:                                                       │
│     -  Only for INDEPENDENT tasks                                │
│     -  Watch API rate limits                                     │
│     -  Add per-chain error handling                              │
│     -  Slowest chain = total time                                │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "If steps don't depend on each other, run them in parallel.     │
│   Your users will thank you for the speed boost."                │
└──────────────────────────────────────────────────────────────────┘

"""